# Week 10 guided practice
CML3113. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and keep its files together. For file-reading examples, upload the supplied CSV into the Colab runtime before running the file-reading example. From the file-reading lesson onward, worked table examples use the supplied CSV. Earlier syntax examples introduce Python before file access.


## Today’s learning
*PowerPoint slide 2*

Use pandas tables for this week’s data work. Explain the decisions and check the outputs.

Complete the issued function interfaces and written evidence using the demonstrated workflow.




## The supplied file
*PowerPoint slide 3*

Extract this week’s student ZIP. Upload engineering_data.csv into Colab using the Files panel, or keep it beside the notebook locally.

The examples below read this file. Run the cells in slide order. Keep the original CSV unchanged.




## Operations for the next example
*PowerPoint slide 4*

Import and short names: import pandas as pd loads pandas and gives it the short name pd. pd.read_csv refers to a function in that library.

Reading a CSV: pd.read_csv("practice_filename.csv") reads the named file into a DataFrame: a table with labelled columns and row labels. The file must exist in the working folder.

Counting dimensions: equipment_data.shape gives (number of rows, number of columns). shape is a property, so it has no parentheses.




## Operations for the next example
*PowerPoint slide 5*

Column data types: equipment_data.dtypes reports each column’s stored type. int64 means integers, float64 means decimal numbers, and object often contains text. Numeric blanks commonly appear as NaN.

A preview of rows: equipment_data.head() returns the first five rows. It leaves the table unchanged. A preview is not the entire dataset.




## Reading the code syntax
*PowerPoint slide 6*

= assigns a value to a name; == compares values. Parentheses pass inputs or continue an expression across lines. print displays a value; len counts entries. int and float convert to integer and decimal numeric values.




## Loading the supplied observations
*PowerPoint slide 7*



Compare the printed result with the explanation in the notebook.


In [ ]:
import pandas as pd

raw_data = pd.read_csv("engineering_data.csv")
print(raw_data.head())
print(raw_data.shape)
print(raw_data.dtypes)


**Example output** (timings and display formatting may vary)

```text
record_id equipment_id  load_pct  ambient_c  target
0          1           E1     59.15      31.71  14.001
1          2           E2     77.46      29.97  15.147
2          3           E3     69.60      19.87  13.005
3          4           E4     75.98      22.29     NaN
4          5           E5     57.62      23.40  12.554
(91, 5)
record_id         int64
equipment_id     object
load_pct        float64
ambient_c       float64
target          float64
dtype: object
```


## Three sets have different jobs
*PowerPoint slide 8*


See the illustration on this PowerPoint slide.

Which set must stay sealed until choices are fixed?


## Rows used for modelling
*PowerPoint slide 9*

Keep the first occurrence of each record_id. Retain recorded targets and load_pct within 0–100. Then separate E1–E4 for training, E5 for validation and E6 for the final test_data.

Fill missing input values using training data only. Keep final-test targets out of model selection.




## Operations for the next example
*PowerPoint slide 10*

Reading chained operations: Read a chain from left to right. In equipment_data["target"].mean(), select the target Series first, then calculate its mean. Each step uses the previous step’s result.

A separate working table: equipment_data.copy() creates a working copy. Assigning columns on this copy preserves the source table, so you can compare before and after.

Finding recorded cells: notna() returns True where a value is present. Use it to retain observations with a recorded target before modelling.




## Operations for the next example
*PowerPoint slide 11*

An inclusive range: series.between(0, 100) is True for values from 0 through 100, including both limits. The resulting mask helps select valid rows.

Selecting columns: equipment_data["column"] returns one labelled column, a Series. equipment_data[["a", "b"]] returns a DataFrame containing those named columns in that order.

Selecting rows with a mask: A column comparison produces one True or False per row. equipment_data[mask] keeps the True rows. & combines masks: both conditions must hold. Parenthesise each comparison.




## Operations for the next example
*PowerPoint slide 12*

Keeping the first observation: equipment_data.drop_duplicates("record_id") returns rows with the first occurrence of each ID, preserving their order. It does not overwrite the source unless you assign the result.

Membership in a group list: series.isin(["E1", "E2"]) returns True for entries matching either listed label. equipment_data[that_mask] keeps all observations belonging to those groups.




## Prepare valid observations
*PowerPoint slide 13*

First keep the first record ID. Then ask two questions for each remaining row: is the target recorded, and is load between 0 and 100? Only a row passing both checks stays. Missing inputs are handled later using training data only.




## Preparing the supplied modelling rows — step 1
*PowerPoint slide 14*



Compare the printed result with the explanation in the notebook.


In [ ]:
modelling_data = raw_data.drop_duplicates("record_id").copy()
target_is_recorded = modelling_data["target"].notna()
load_is_valid = modelling_data["load_pct"].between(0, 100)
rows_to_keep = target_is_recorded & load_is_valid
print(target_is_recorded.head())
print(load_is_valid.head())
modelling_data = modelling_data[rows_to_keep].copy()
print(len(raw_data), len(modelling_data))


**Example output** (timings and display formatting may vary)

```text
0     True
1     True
2     True
3    False
4     True
Name: target, dtype: bool
0    True
1    True
2    True
3    True
4    True
Name: load_pct, dtype: bool
91 88
```


## Choose groups before choosing rows
*PowerPoint slide 15*

isin(list_of_labels) answers True when a row’s equipment ID appears in the list. Here E1–E4 supply training rows, E5 validation rows and E6 test rows. The groups stay separate so one equipment group cannot appear on both sides.




## Preparing the supplied modelling rows — step 2
*PowerPoint slide 16*



Compare the printed result with the explanation in the notebook.


In [ ]:
is_training_equipment = modelling_data["equipment_id"].isin(
    ["E1", "E2", "E3", "E4"]
)
is_validation_equipment = modelling_data["equipment_id"].isin(
    ["E5"]
)
is_test_equipment = modelling_data["equipment_id"].isin(["E6"])


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Preparing the supplied modelling rows — step 3
*PowerPoint slide 17*

training_data: rows used to learn model settings; only these rows may determine imputation values.

validation_data: separate rows used to compare models without fitting on them.




## Preparing the supplied modelling rows — step 3
*PowerPoint slide 18*



Compare the printed result with the explanation in the notebook.


In [ ]:
training_data = modelling_data[is_training_equipment].copy()
validation_data = modelling_data[is_validation_equipment].copy()
test_data = modelling_data[is_test_equipment].copy()
feature_columns = ["load_pct", "ambient_c"]
print(len(training_data), len(validation_data), len(test_data))


**Example output** (timings and display formatting may vary)

```text
58 15 15
```


## Training-only filling
*PowerPoint slide 19*

median ignores missing values. Calculate it from training input values and reuse it for validation gaps.

Do not calculate a new median from validation or test observations. A pipeline keeps this rule with the linear_model.




## Operations for the next example
*PowerPoint slide 20*

Middle recorded value: series.median() sorts known values conceptually and returns the middle one. With an even count, it averages the two middle values. Missing entries do not enter the calculation.

Replacing missing entries: series.fillna(value) returns a column whose missing entries contain value. Recorded values stay unchanged. Assign the returned column to retain the fill.

Rounded display: series.round(3) returns values rounded to three decimal places. Python round(number, 3) rounds one number. Keep unrounded calculations for returned assessment answers.




## Operations for the next example
*PowerPoint slide 21*

Adding values and counting True: series.sum() adds its values. For a Boolean mask, True counts as 1 and False as 0. equipment_data.isna().sum() therefore counts missing cells separately for each column.

Finding missing cells: isna() returns True for missing entries such as NaN and None. It returns False for a recorded zero. A DataFrame gives a mask for every cell; a Series gives one mask value per row.




## Training median from the actual ambient readings
*PowerPoint slide 22*



Compare the printed result with the explanation in the notebook.


In [ ]:
training_ambient_median = training_data["ambient_c"].median()
filled_validation_ambient = validation_data["ambient_c"].fillna(
    training_ambient_median
)
print(round(float(training_ambient_median), 3))
print(validation_data["ambient_c"].isna().sum())
print(filled_validation_ambient.head())


**Example output** (timings and display formatting may vary)

```text
26.07
0
4     23.40
10    33.98
16    34.28
22    30.81
28    21.33
Name: ambient_c, dtype: float64
```


## Training a model with missing inputs
*PowerPoint slide 23*

training_data[features] selects the two input columns as a equipment_data. training_data["target"] selects the observed target as a Series.

SimpleImputer(strategy="median") learns input fill values from training rows. make_pipeline links that operation to LinearRegression. fit learns the steps and predict reuses them.




## Operations for the next example
*PowerPoint slide 24*

Returning plain values: series.tolist() or array.tolist() returns a Python list of values, without pandas row labels. Preserve their order when pairing actual values and validation_predictions.




## A sequence of model steps / Filling model inputs
*PowerPoint slide 25*

make_pipeline(imputer, estimator) links preprocessing and prediction. fit learns the imputer and estimator on training rows. predict applies the learned steps to new rows in the same order.

SimpleImputer(strategy="median") learns a median for each input column during fit. It uses those training medians to fill gaps later. Never learn replacement values from validation or test rows.




## Creating linear regression / Learning from training examples
*PowerPoint slide 26*

LinearRegression() creates an unfitted model. It will learn coefficients and an intercept from training examples. Creating it alone does not train it or make predictions.

model.fit(input_table, observed_targets) learns model settings from paired rows. The two inputs must describe the same observations in the same order. Call fit before predict.




## Predicting new targets / Selecting a short preview
*PowerPoint slide 27*

model.predict(input_table) returns one prediction per input row. Use the same feature columns and order as during training. Prediction uses learned settings; it does not refit the model.

values[:5] selects positions 0 through 4, or fewer if the sequence is shorter. The stop position is excluded. This preview does not change the full array stored in the variable.




## Standard names for model inputs
*PowerPoint slide 28*

X_train: training input columns. y_train: the matching observed targets.

X_valid: validation input columns. y_valid: their observed targets, kept for evaluation.

Capital X usually means a two-dimensional table; lowercase y usually means one target per row.




## Separate inputs from answers
*PowerPoint slide 29*

X_train contains the input columns; y_train contains their observed targets. X_valid contains the same input columns for validation; y_valid holds the answers for checking predictions. Never include target among the inputs.




## Training and predicting on the supplied data — step 1
*PowerPoint slide 30*



Compare the printed result with the explanation in the notebook.


In [ ]:
X_train = training_data[feature_columns]
y_train = training_data["target"]
X_valid = validation_data[feature_columns]
y_valid = validation_data["target"]
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Create, then train, then predict
*PowerPoint slide 31*

Creating an estimator sets up an empty model. fit learns settings from training inputs and their targets. predict uses those learned settings on new inputs; it does not learn from the validation targets.




## Training and predicting on the supplied data — step 2
*PowerPoint slide 32*



Compare the printed result with the explanation in the notebook.


In [ ]:
linear_model = make_pipeline(
    SimpleImputer(strategy="median"), LinearRegression()
)
linear_model.fit(X_train, y_train)
validation_predictions = linear_model.predict(X_valid)
print(validation_predictions[:5].round(3).tolist())


**Example output** (timings and display formatting may vary)

```text
[11.79, 9.69, 14.015, 8.537, 10.471]
```


## Each fold takes a turn at validation
*PowerPoint slide 33*


See the illustration on this PowerPoint slide.

Illustrated spread: population SD 0.39 MPa, not a confidence interval.


## Cross-validation within training data
*PowerPoint slide 34*

GroupKFold keeps each equipment group within one fold. Each fold takes a turn as validation while the remaining folds train the pipeline.

cross_val_score reports one score per fold. neg_mean_absolute_error returns negative MAE, so negate the scores before reporting baseline_errors.




## Operations for the next example
*PowerPoint slide 35*

A labelled column: pd.Series(values) creates a one-dimensional labelled sequence. With an array of fold errors, the default labels are 0, 1, 2 and so on.

Mean of recorded values: series.mean() adds known values and divides by their count. pandas skips missing values by default. The result has the same unit as the original column.

Spread of fold errors: series.std(ddof=0) measures how far values spread around their mean. ddof=0 divides squared deviations by the number of values. This describes these folds, not a confidence interval.




## Keeping groups together / Scoring repeated splits
*PowerPoint slide 36*

GroupKFold(n_splits=4) creates four train/validation splits. All rows of an equipment group stay on one side of each split. Four splits require at least four distinct training groups.

cross_val_score fits a fresh model for each split and scores its held-out rows. groups identifies the equipment groups and cv supplies the split rule. neg_mean_absolute_error returns negative MAE; negate it to report positive errors.




## Repeat evaluation without mixing groups
*PowerPoint slide 37*

Four folds means four held-out group turns. In each turn, learn a fresh pipeline on the other training groups and score the held-out group. This uses training_data only; validation and test tables stay outside this calculation.




## Fold errors calculated from the supplied training rows — step 1
*PowerPoint slide 38*



Compare the printed result with the explanation in the notebook.


In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score

grouped_folds = GroupKFold(n_splits=4)


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Read the scoring convention
*PowerPoint slide 39*

scikit-learn uses larger scores as better. neg_mean_absolute_error therefore returns negative errors. Negate the returned scores to recover ordinary positive MAE; lower MAE is better.




## Fold errors calculated from the supplied training rows — step 2
*PowerPoint slide 40*



Compare the printed result with the explanation in the notebook.


In [ ]:
negative_error_scores = cross_val_score(
    linear_model,
    X_train,
    y_train,
    groups=training_data["equipment_id"],
    cv=grouped_folds,
    scoring="neg_mean_absolute_error",
)


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Report typical error and variation
*PowerPoint slide 41*

Each value is the MAE of one held-out fold. mean gives average error. std(ddof=0) describes spread across these four errors; it is not a confidence interval. A large spread suggests performance depends on the held-out equipment.




## Fold errors calculated from the supplied training rows — step 3
*PowerPoint slide 42*



Compare the printed result with the explanation in the notebook.


In [ ]:
fold_absolute_errors = pd.Series(-negative_error_scores)
print(fold_absolute_errors.round(4).tolist())
print(round(float(fold_absolute_errors.mean()), 4))
print(round(float(fold_absolute_errors.std(ddof=0)), 4))


**Example output** (timings and display formatting may vary)

```text
[0.5657, 0.7333, 0.7221, 0.7835]
0.7012
0.0815
```


## Pandas inside the required function
*PowerPoint slide 43*

The assignment caller supplies equipment_records. pd.DataFrame(records) gives those records named columns. Calculate with pandas, then return the published Python type.

to_dict("records") makes a list of row dictionaries. Series.tolist() makes a list of values. A grouped Series.to_dict() maps group labels to validation_results.




## Operations for the next example
*PowerPoint slide 44*

Returning a dictionary: series.to_dict() maps Series labels to values. equipment_data.to_dict("records") instead returns a list of row dictionaries. Choose the form required by the assignment.

A table from Python values: pd.DataFrame(records) builds a table from row dictionaries. With a dictionary of column lists, keys become column names and each list supplies that column’s values.

Grouping rows: equipment_data.groupby("label")["value"] groups the value column by matching labels. Apply mean() or sum() to calculate one result per group. Missing group labels are excluded by default.




## Defining and calling a function
*PowerPoint slide 45*

def gives a reusable block a name and input parameters. The indented statements run when it is called. return sends a result back to the caller; print displays a result but does not return it.




## A function called with the supplied records
*PowerPoint slide 46*



Compare the printed result with the explanation in the notebook.


In [ ]:
def mean_load_by_equipment(equipment_records):
    equipment_data = pd.DataFrame(equipment_records)
    mean_load_by_equipment_group = equipment_data.groupby(
        "equipment_id"
    )["load_pct"].mean()
    return mean_load_by_equipment_group.to_dict()


equipment_records = raw_data.to_dict("records")
print(mean_load_by_equipment(equipment_records))


**Example output** (timings and display formatting may vary)

```text
{'E1': 54.9475, 'E2': 61.326, 'E3': 53.46933333333333, 'E4': 66.854, 'E5': 48.10133333333333, 'E6': 51.54}
```


## Continue with your issued assignment
Open the assignment notebook named in 00_START_HERE.txt. Keep required function names, return types and units. guided_practice.ipynb is not a graded upload. Use SUBMIT_HERE.txt for the correct submission link.
